In [ ]:
import pandas as pd
import numpy as np

# Load datasets safely
amazon = pd.read_csv('Amazon Sale Report.csv', low_memory=False)
products = pd.read_csv('Sale Report.csv', low_memory=False)
pricing = pd.read_csv('P L March 2021.csv', low_memory=False)
intl = pd.read_csv('International sale Report.csv', low_memory=False)

# -------------------------------
# 🧹 CLEAN COLUMN NAMES
# -------------------------------
def clean_columns(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

amazon = clean_columns(amazon)
products = clean_columns(products)
pricing = clean_columns(pricing)
intl = clean_columns(intl)

# -------------------------------
# 🔄 FIX DATA TYPES
# -------------------------------

# Amazon dataset
amazon['amount'] = pd.to_numeric(amazon['amount'], errors='coerce')
amazon['qty'] = pd.to_numeric(amazon['qty'], errors='coerce')
amazon['date'] = pd.to_datetime(amazon['date'], errors='coerce')

# International dataset
intl['gross_amt'] = pd.to_numeric(intl['gross_amt'], errors='coerce')
intl['pcs'] = pd.to_numeric(intl['pcs'], errors='coerce')
intl['date'] = pd.to_datetime(intl['date'], errors='coerce')

# -------------------------------
# 🗑️ REMOVE BAD DATA
# -------------------------------
amazon = amazon.dropna(subset=['amount', 'qty'])
intl = intl.dropna(subset=['gross_amt', 'pcs'])

# -------------------------------
# 🔍 QUICK CHECK
# -------------------------------
print("Amazon Data:")
print(amazon.info())

print("\nProducts Data:")
print(products.info())

print("\nPricing Data:")
print(pricing.info())

print("\nInternational Data:")
print(intl.info())

/tmp/ipykernel_11554/3219915863.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon['date'] = pd.to_datetime(amazon['date'], errors='coerce')
/tmp/ipykernel_11554/3219915863.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  intl['date'] = pd.to_datetime(intl['date'], errors='coerce')


Amazon Data:
<class 'pandas.core.frame.DataFrame'>
Index: 121180 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   index               121180 non-null  int64         
 1   order_id            121180 non-null  object        
 2   date                121180 non-null  datetime64[ns]
 3   status              121180 non-null  object        
 4   fulfilment          121180 non-null  object        
 5   sales_channel       121180 non-null  object        
 6   ship-service-level  121180 non-null  object        
 7   style               121180 non-null  object        
 8   sku                 121180 non-null  object        
 9   category            121180 non-null  object        
 10  size                121180 non-null  object        
 11  asin                121180 non-null  object        
 12  courier_status      116044 non-null  object        
 13  qty                 1

In [ ]:
amazon.columns
products.columns
pricing.columns
intl.columns

Index(['index', 'date', 'months', 'customer', 'style', 'sku', 'size', 'pcs',
       'rate', 'gross_amt'],
      dtype='object')

In [ ]:
intl = intl.drop(columns=['index'])

In [ ]:
amazon.columns
products.columns
pricing.columns
intl.columns

Index(['date', 'months', 'customer', 'style', 'sku', 'size', 'pcs', 'rate',
       'gross_amt'],
      dtype='object')

In [ ]:
products.columns
pricing.columns
amazon.columns

Index(['index', 'order_id', 'date', 'status', 'fulfilment', 'sales_channel',
       'ship-service-level', 'style', 'sku', 'category', 'size', 'asin',
       'courier_status', 'qty', 'currency', 'amount', 'ship-city',
       'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids',
       'b2b', 'fulfilled-by', 'unnamed:_22'],
      dtype='object')

In [ ]:
# Drop useless columns
amazon = amazon.drop(columns=['index', 'unnamed:_22'], errors='ignore')

# Fix column names (remove hyphens)
amazon.columns = amazon.columns.str.replace('-', '_')

In [ ]:
amazon['amount'] = pd.to_numeric(amazon['amount'], errors='coerce')
amazon['qty'] = pd.to_numeric(amazon['qty'], errors='coerce')
amazon['date'] = pd.to_datetime(amazon['date'], errors='coerce')

In [ ]:
amazon = amazon.dropna(subset=['amount', 'qty'])

In [ ]:
amazon.columns
products.columns
pricing.columns
intl.columns

Index(['date', 'months', 'customer', 'style', 'sku', 'size', 'pcs', 'rate',
       'gross_amt'],
      dtype='object')

In [ ]:
intl['pcs'] = pd.to_numeric(intl['pcs'], errors='coerce')
intl['rate'] = pd.to_numeric(intl['rate'], errors='coerce')
intl['gross_amt'] = pd.to_numeric(intl['gross_amt'], errors='coerce')
intl['date'] = pd.to_datetime(intl['date'], errors='coerce')

In [ ]:
intl = intl.dropna(subset=['pcs', 'gross_amt'])

In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')

In [ ]:
amazon.to_sql('amazon_sales', conn, index=False, if_exists='replace')
products.to_sql('products', conn, index=False, if_exists='replace')
intl.to_sql('international_sales', conn, index=False, if_exists='replace')

36391

In [ ]:
products.columns

Index(['index', 'sku_code', 'design_no.', 'stock', 'category', 'size',
       'color'],
      dtype='object')

In [ ]:
products.rename(columns={'sku_code': 'sku'}, inplace=True)

In [ ]:
products.to_sql('products', conn, index=False, if_exists='replace')

9271

In [ ]:
query = """
SELECT
    p.category,
    SUM(i.gross_amt) AS international_revenue
FROM international_sales i
JOIN products p
ON i.sku = p.sku
GROUP BY p.category
ORDER BY international_revenue DESC;
"""

In [ ]:
result = pd.read_sql(query, conn)
result

,category,international_revenue
0,KURTA SET,5206028.0
1,KURTA,4879920.0
2,SET,1284146.0
3,TOP,1094561.0
4,DRESS,937709.0
5,TUNIC,367457.0
6,PANT,228894.0
7,PALAZZO,218461.0
8,NIGHT WEAR,101319.0
9,LEHENGA CHOLI,82876.0


In [ ]:
query = """
SELECT
    p.category,
    SUM(i.gross_amt) AS revenue,
    RANK() OVER (ORDER BY SUM(i.gross_amt) DESC) AS rank
FROM international_sales i
JOIN products p
ON i.sku = p.sku
GROUP BY p.category;
"""

In [ ]:
result = pd.read_sql(query, conn)
result

,category,revenue,rank
0,KURTA SET,5206028.0,1
1,KURTA,4879920.0,2
2,SET,1284146.0,3
3,TOP,1094561.0,4
4,DRESS,937709.0,5
5,TUNIC,367457.0,6
6,PANT,228894.0,7
7,PALAZZO,218461.0,8
8,NIGHT WEAR,101319.0,9
9,LEHENGA CHOLI,82876.0,10


In [ ]:
query = """
SELECT
    p.category,
    SUM(i.gross_amt) AS revenue,
    ROUND(100.0 * SUM(i.gross_amt) /
        (SELECT SUM(gross_amt) FROM international_sales), 2) AS percentage,
    RANK() OVER (ORDER BY SUM(i.gross_amt) DESC) AS rank
FROM international_sales i
JOIN products p
ON i.sku = p.sku
GROUP BY p.category
ORDER BY revenue DESC;
"""

In [ ]:
result = pd.read_sql(query, conn)
result

,category,revenue,percentage,rank
0,KURTA SET,5206028.0,31.75,1
1,KURTA,4879920.0,29.76,2
2,SET,1284146.0,7.83,3
3,TOP,1094561.0,6.68,4
4,DRESS,937709.0,5.72,5
5,TUNIC,367457.0,2.24,6
6,PANT,228894.0,1.40,7
7,PALAZZO,218461.0,1.33,8
8,NIGHT WEAR,101319.0,0.62,9
9,LEHENGA CHOLI,82876.0,0.51,10


In [ ]:
result

,category,revenue,percentage,rank
0,KURTA SET,5206028.0,31.75,1
1,KURTA,4879920.0,29.76,2
2,SET,1284146.0,7.83,3
3,TOP,1094561.0,6.68,4
4,DRESS,937709.0,5.72,5
5,TUNIC,367457.0,2.24,6
6,PANT,228894.0,1.40,7
7,PALAZZO,218461.0,1.33,8
8,NIGHT WEAR,101319.0,0.62,9
9,LEHENGA CHOLI,82876.0,0.51,10


In [ ]:
result.to_csv('category_analysis.csv', index=False)